# 1. RAG 原理与工作流程

来源：《尚硅谷-10-RAG.pdf》，第 1–7 页；对应课件 **1.1–1.4**。页码均指 PDF 阅读器中的页序。

整理方式：原理保留关键逻辑；第 2 章示例精简，重点学习拆分器源码与递归拆分。
只记录到 2.5.2，不收录 2.5.3 案例。

**运行约定**：在 `D:/grow/langchain/7_RAG` 打开本文件，选择
`D:/conda/envs/langchain/python.exe` 对应的 Python 内核；代码单元按本文件顺序运行。
各 Notebook 的变量相互独立。使用 `len` 的示例以 Python 字符长度计数，不能据此推断 Token 数。

## 1.1 为什么需要 Retrieval

大模型的主要局限：**知识滞后、知识缺失、幻觉**。企业内部文档、个人文件及训练之后产生的数据，
通常不能仅靠模型自身的参数知识回答。

RAG（Retrieval-Augmented Generation，检索增强生成）的核心是：
**先从外部资料中检索相关内容，再把内容和问题一起交给模型生成回答。**
它补充模型本次回答的上下文，不会因此直接更新模型权重，也不能保证完全消除幻觉。

## 1.2 与其他方式的关系

| 方式 | 主要做什么 | 在本章中的理解 |
| --- | --- | --- |
| 提示词工程 | 明确任务、规则、输出格式，并提供上下文 | RAG 检索出的资料最终也需要组织进提示词 |
| RAG | 按问题检索外部知识 | 适合需要资料依据、私有知识或知识更新的任务 |
| 微调 | 训练并改变模型参数 | 与 RAG 可以配合使用；不等于给模型挂一个可更新的文档库 |

课件强调的优点：更丰富的资料背景、便于更新知识、回答可以追溯到来源。
主要代价：增加检索延迟，检索内容占用上下文和 Token，并需要维护知识数据。
业务数据是否保持私有还取决于部署方式和调用的服务，RAG 本身不是隐私保证。

## 1.3 两条流程：先建索引，再回答问题

```text
建索引：Source → Load → Transform / Split → Embed → Store
         数据源    Document      文本块        向量     向量存储

问答：用户问题 → 用匹配的 Embedding 模型编码 → Retrieve
                                               ↓
                         问题 + 检索文本 + 提示词 → LLM → 回答
```

| 环节 | 输入 → 输出 | 需要抓住的事 |
| --- | --- | --- |
| Source | 文件、网页、API 等 → 原始内容 | 资料可以是文本、表格，也可以先从图片、音视频中提取文本 |
| Load | 原始内容 → `Document` | 统一为 `page_content` 与 `metadata` |
| Transform | `Document` → 处理后的 `Document` | 可以清洗、去重、提取元数据；本章重点是拆分 |
| Split | 长文档 → 多个 Chunk | 兼顾长度、边界和后续检索粒度 |
| Embed | `str` / `list[str]` → 向量 / 向量列表 | 文档和查询必须进入相容的向量空间 |
| Store | 文本块、向量、元数据 → 索引 | 保留原文和来源，不能只留下无法解释的数值 |
| Retrieve | 问题 → 相关 `Document` | 检索得到的是资料，还需要组织上下文并生成回答 |

**补充理解**：课件以长文档为背景强调拆分；如果资料本身很短且满足模型限制，并非必须先切一刀。
文本拆分也不等于语义理解。后续需要检查：原文提取是否正确、检索内容是否相关、回答是否有依据。

## 1.4 本章复习问题

- `Document`、Chunk、Embedding、向量库分别保存什么？
- 为什么用户问题也需要向量化？为什么不能随便更换查询模型？
- 只把文件内容拼进 Prompt，与先检索相关片段相比，有什么成本和长度差异？
- 为什么分块太小可能缺少上下文，太大又可能带入噪声？

下一份笔记从统一的数据对象 `Document` 开始。

## 笔记导航

1. [7_1_RAG原理与流程](7_1_RAG原理与流程.ipynb)
2. [7_2_Document与文档加载](7_2_Document与文档加载.ipynb)
3. [7_3_TextSplitter源码与Character切分](7_3_TextSplitter源码与Character切分.ipynb)
4. [7_4_RecursiveCharacterTextSplitter详解](7_4_RecursiveCharacterTextSplitter详解.ipynb)
5. [7_5_其他拆分方式速记](7_5_其他拆分方式速记.ipynb)
6. [7_6_Embedding与向量存储](7_6_Embedding与向量存储.ipynb)